# Component_01 — STAGE 4B: Decoding Ablation
### No retraining. Load `best.pt`, change only how it decodes.

---

## Why this exists

Two loose ends from Stage 4.

### 1. Beam search may be hurting you

| | decoder | ROUGE-L | unique firsts |
|---|---|---|---|
| val, epoch 10 | **greedy** | **0.3009** | **0.22** |
| test | **beam-4** | 0.2861 | 0.1065 |

Different sets, so not conclusive — but beam search is known to concentrate
probability mass and suppress diversity. If greedy wins on test too, your margin
over the constant baseline goes from **+0.0092** toward **+0.02** for one compute unit.

### 2. My diversity comparison was invalid — and I need to fix it

`unique_firsts = len(set(openings)) / n`

- OLD model measured at **n = 100** → 14 unique → 0.14
- NEW model measured at **n = 4,722** → 503 unique → 0.1065

**503 unique openings versus 14** — but the ratio *looks* worse because the
denominator is 47× larger. That was my error in the Stage 4 notebook.

This notebook fixes it two ways:

- reports every diversity metric at **matched n** (100 / 500 / 1000 / full)
- measures the **reference reports** on the identical metric at the identical n,
  which is the only honest ceiling. "Good" is not 1.0 — it is whatever real
  radiologists score.

---

## Cost

Strategies are compared on a **1,000-sample subset**, then the winner runs on the
**full test set**. ≈25–30 min ≈ **1 CU**. No training, no gradients.

---
# 0 · Config

In [ ]:
CFG = dict(
    IMG_SIZE=384, MAX_TOKENS=256, NUM_WORKERS=4,
    DECODER="GanjinZero/biobart-v2-base",
    UNFREEZE_VISION_STAGES=1, PROJ_DROPOUT=0.1,
    GEN_MAX_TOKENS=192, GEN_MIN_TOKENS=24,
    ABLATION_N=1000,          # subset for comparing strategies
    AMP_DTYPE="bf16", SEED=42,
)
CONST_BASELINE_ROUGEL = 0.2769
STAGE4_TEST = dict(rougeL=0.2861, margin=0.0092, uniq_firsts=0.1065,
                   uniq_reports=0.5813, vocab_ratio=0.2063, clinical_f1=0.5648,
                   gen_words=41.15)     # what Stage 4 reported, for reference

# Strategies to compare. `kw` goes straight to model.generate().
STRATEGIES = [
    ("greedy",              dict(num_beams=1)),
    ("beam2_lp1.0",         dict(num_beams=2, length_penalty=1.0, early_stopping=True)),
    ("beam4_lp1.2 (stage4)",dict(num_beams=4, length_penalty=1.2, early_stopping=True)),
    ("beam4_lp1.0",         dict(num_beams=4, length_penalty=1.0, early_stopping=True)),
    ("nucleus_p0.9",        dict(do_sample=True, top_p=0.9, temperature=0.9, num_beams=1)),
    # NOT diverse/group beam search: it was moved out of core transformers in v5
    # and now needs trust_remote_code plus a remote download — verified to raise
    # ValueError on the current build. Conservative top-k sampling probes the same
    # diversity question with no extra dependency.
    ("topk50_t0.7",         dict(do_sample=True, top_k=50, temperature=0.7, num_beams=1)),
]
print(f"{len(STRATEGIES)} strategies on {CFG['ABLATION_N']} samples, then the winner on the full test set")

---
# 1 · Environment

In [ ]:
import os, sys, json, time, random, subprocess, re, gc, warnings, importlib
warnings.filterwarnings("ignore")
from pathlib import Path
from datetime import datetime
import numpy as np

try:
    GPU = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=15).stdout.strip()
except Exception:
    GPU = ""
if not GPU:
    raise SystemExit("No GPU. Runtime → Change runtime type → L4.")
print(f"  GPU: {GPU}")
RATE = 1.75 if "L4" in GPU else 2.0
need = [p for m, p in [("transformers", "transformers"), ("rouge_score", "rouge-score")]
        if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

import torch, torch.nn as nn, pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from transformers import AutoTokenizer, BartForConditionalGeneration
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
CFG["NUM_WORKERS"] = max(2, min(CFG["NUM_WORKERS"], os.cpu_count() or 4))
DEV = torch.device("cuda")
AMP_DT = torch.bfloat16 if CFG["AMP_DTYPE"] == "bf16" else torch.float16
T0 = time.time()
print(f"  torch {torch.__version__} | workers {CFG['NUM_WORKERS']}")

---
# 2 · Drive, images, model

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
PROJECT  = Path("/content/drive/MyDrive/Component_01")
MANIFEST = PROJECT / "training_manifest"
S4_BEST  = PROJECT / "checkpoints" / "stage4" / "best.pt"
S5_CKPT  = PROJECT / "checkpoints" / "stage5" / "backbone_for_stage4.pt"
REPORTS  = PROJECT / "reports" / "stage4b"; REPORTS.mkdir(parents=True, exist_ok=True)
IMG_ROOT = Path("/content/cardio_image_384")
TAR      = PROJECT / "data" / "images" / "cardio_384.tar"

# copy-then-extract: streaming tar straight off Drive has failed before
if not IMG_ROOT.exists():
    LOCAL = Path("/content/cardio_384.tar")
    if not LOCAL.exists() or LOCAL.stat().st_size != TAR.stat().st_size:
        t = time.time(); import shutil; shutil.copy(TAR, LOCAL)
        print(f"  copied tar in {time.time()-t:.0f}s")
    r = subprocess.run(["tar", "-xf", str(LOCAL), "-C", "/content/"], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[:400]
    LOCAL.unlink()
n_png = sum(1 for _ in IMG_ROOT.rglob("*.png"))
assert n_png >= 45000, f"only {n_png} images"
print(f"  images: {n_png:,}")
assert S4_BEST.exists(), f"{S4_BEST} missing — run Stage 4 first"

TOK = AutoTokenizer.from_pretrained(CFG["DECODER"])
TEST = pd.read_csv(MANIFEST / "manifest_test.csv", low_memory=False)
print(f"  test rows: {len(TEST):,}")

---
# 3 · Model (identical definition to Stage 4 — state_dict must match)

In [ ]:
_EPS = 1e-6
class ToGrayscalePIL:
    def __call__(self, img): return img if img.mode == "L" else img.convert("L")
class PerImageZScore:
    def __init__(self, c=3): self.c = c
    def __call__(self, t):
        if t.shape[0] != 1: t = t[:1]
        s = t.std(); t = (t - t.mean())/s if s > _EPS else t - t.mean()
        return t.repeat(self.c, 1, 1)
EVAL_TF = transforms.Compose([ToGrayscalePIL(), transforms.Resize((CFG["IMG_SIZE"],)*2),
                              transforms.ToTensor(), PerImageZScore(3)])

class CXRReportGenerator(nn.Module):
    def __init__(self, backbone_path, decoder_name, unfreeze_stages=1):
        super().__init__()
        self.vision = models.convnext_base(weights=None).features
        ck = torch.load(backbone_path, map_location="cpu", weights_only=False)
        feat = ck["features"] if "features" in ck else ck
        self.vision.load_state_dict({k.replace("features.", ""): v for k, v in feat.items()},
                                    strict=False)
        for p in self.vision.parameters(): p.requires_grad = False
        self.unfrozen = []
        if unfreeze_stages > 0:
            for m in list(self.vision.children())[-unfreeze_stages*2:]:
                for p in m.parameters(): p.requires_grad = True
                self.unfrozen.append(m)
        self.bart = BartForConditionalGeneration.from_pretrained(decoder_name)
        d = self.bart.config.d_model
        self.proj = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d), nn.GELU(),
                                  nn.Dropout(CFG["PROJ_DROPOUT"]), nn.Linear(d, d), nn.LayerNorm(d))
    def encode(self, x):
        with torch.no_grad():
            f = self.vision(x)
        return self.proj(f.flatten(2).transpose(1, 2))
    @torch.no_grad()
    def generate(self, x, **kw):
        e = self.encode(x); m = torch.ones(e.shape[:2], dtype=torch.long, device=e.device)
        enc = self.bart.model.encoder(inputs_embeds=e, attention_mask=m)
        return self.bart.generate(encoder_outputs=enc, attention_mask=m,
                                  max_length=CFG["GEN_MAX_TOKENS"],
                                  min_length=CFG["GEN_MIN_TOKENS"],
                                  no_repeat_ngram_size=3, **kw)

model = CXRReportGenerator(S5_CKPT, CFG["DECODER"], CFG["UNFREEZE_VISION_STAGES"]).to(DEV)
ck = torch.load(S4_BEST, map_location="cpu", weights_only=False)
missing, unexpected = model.load_state_dict(ck["model"], strict=False)
assert not unexpected, f"unexpected keys: {list(unexpected)[:5]}"
# Stage 4 selected and reported on EMA weights — replicate that exactly
if ck.get("ema"):
    sd = model.state_dict()
    for k, v in ck["ema"].items():
        if k in sd: sd[k].copy_(v.to(sd[k].device))
    print("  EMA weights applied (matches Stage 4's reported model)")
model.eval()
print(f"  loaded best.pt from epoch {ck['epoch']} (val ROUGE-L {ck['best_metric']:.4f})")

---
# 4 · Data

In [ ]:
class TestDS(Dataset):
    def __init__(self, df):
        self.p = [str(IMG_ROOT / x) for x in df["image_path"]]
        self.r = df["report"].astype(str).tolist()
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return EVAL_TF(Image.open(self.p[i])), self.r[i]

def collate(b):
    im, tx = zip(*b); return torch.stack(im), list(tx)

FULL_DS = TestDS(TEST)
rng = np.random.RandomState(CFG["SEED"])
SUB_IDX = sorted(rng.choice(len(FULL_DS), min(CFG["ABLATION_N"], len(FULL_DS)), replace=False).tolist())
SUB_DS = torch.utils.data.Subset(FULL_DS, SUB_IDX)
print(f"  ablation subset {len(SUB_DS)} | full test {len(FULL_DS)}")

BATCH = 16
for bs in (32, 24, 16, 8):
    try:
        torch.cuda.empty_cache()
        x = torch.randn(bs, 3, 384, 384, device=DEV)
        with torch.autocast("cuda", dtype=AMP_DT):
            _ = model.generate(x, num_beams=4, length_penalty=1.0, early_stopping=True)
        BATCH = bs; del x; torch.cuda.empty_cache(); print(f"  batch {bs}: OK"); break
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        if "out of memory" not in str(e).lower(): raise
        torch.cuda.empty_cache(); gc.collect(); print(f"  batch {bs}: OOM")
print(f"  using batch {BATCH} (beam-4 worst case)")

---
# 5 · Metrics — with matched-n diversity and a REFERENCE ceiling

In [ ]:
_SC = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
PRIOR_RE = re.compile(
    r"(as compared (to|with)|in compar(ison|ed) (to|with)"
    r"|\b(prior|previous|earlier|preceding)\s+(studies|study|exams?|radiographs?|films?|imaging)"
    r"|\b(unchanged|stable|constant)\b|\bno (significant |relevant |interval )?changes?\b"
    r"|\b(increasing|decreasing|worsening|improving)\b|___"
    r"|\b(again|persistent|persists|remains?)\b)", re.I)
PATH_KW = {
 "Cardiomegaly": r"(cardiomegaly|cardiac enlargement|enlarged cardiac silhouette|heart.{0,20}enlarged)",
 "Edema": r"(pulmonary edema|interstitial edema|\bedema\b|vascular congestion)",
 "Pleural_Effusion": r"(pleural effusion|\beffusions?\b)", "Atelectasis": r"atelecta",
 "Consolidation": r"consolidat", "Lung_Opacity": r"(opacit|infiltrate)",
 "Pneumonia": r"pneumonia", "Pneumothorax": r"pneumothora"}
NEG_RE = re.compile(r"\b(no|not|without|negative for|free of|absence of|absent)\b", re.I)

def assert_labels(t):
    out = {}; sents = re.split(r"(?<=[.;])\s+", re.sub(r"\s+", " ", t or ""))
    for lab, pat in PATH_KW.items():
        kw = re.compile(pat, re.I); pos = 0
        for s in sents:
            for m in kw.finditer(s):
                if not NEG_RE.search(s[:m.start()]): pos = 1; break
            if pos: break
        out[lab] = pos
    return out

def clinical_f1(P, R):
    tp = fp = fn = 0
    for p, r in zip(P, R):
        a, b = assert_labels(p), assert_labels(r)
        for k in PATH_KW:
            tp += a[k] == 1 and b[k] == 1
            fp += a[k] == 1 and b[k] == 0
            fn += a[k] == 0 and b[k] == 1
    pr = tp/max(tp+fp, 1); rc = tp/max(tp+fn, 1)
    return 2*pr*rc/max(pr+rc, 1e-9)

def diversity(texts):
    """Sample-size dependent — ALWAYS compare at equal n."""
    n = max(len(texts), 1)
    firsts = [re.split(r"(?<=[.])\s", t.strip())[0] for t in texts if t.strip()]
    vocab = {w for t in texts for w in t.lower().split()}
    return dict(uniq_firsts=len(set(firsts))/max(len(firsts), 1),
                uniq_reports=len(set(texts))/n,
                vocab=len(vocab),
                words=float(np.mean([len(t.split()) for t in texts])) if texts else 0.0)

def score(P, R):
    r1 = r2 = rl = 0.0
    for p, r in zip(P, R):
        s = _SC.score(r, p); r1 += s["rouge1"].fmeasure; r2 += s["rouge2"].fmeasure; rl += s["rougeL"].fmeasure
    n = max(len(P), 1); d = diversity(P); dr = diversity(R)
    return dict(rouge1=r1/n, rouge2=r2/n, rougeL=rl/n, margin=rl/n - CONST_BASELINE_ROUGEL,
                clinical_f1=clinical_f1(P, R),
                prior_rate=sum(bool(PRIOR_RE.search(p)) for p in P)/n,
                vocab_ratio=d["vocab"]/max(dr["vocab"], 1),
                gen_words=d["words"], ref_words=dr["words"],
                **{k: d[k] for k in ("uniq_firsts", "uniq_reports")})

@torch.no_grad()
def run(ds, gen_kw, tag):
    dl = DataLoader(ds, batch_size=BATCH, shuffle=False, collate_fn=collate,
                    num_workers=CFG["NUM_WORKERS"], pin_memory=True)
    P, R = [], []
    for x, tx in tqdm(dl, desc=f"  {tag}", leave=False):
        x = x.to(DEV, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DT):
            ids = model.generate(x, **gen_kw)
        P += TOK.batch_decode(ids, skip_special_tokens=True); R += list(tx)
    return P, R

---
# 6 · Ablation

In [ ]:
print("=" * 104); print(f"  DECODING ABLATION  ({len(SUB_DS)} test samples)"); print("=" * 104)
print(f"  {'strategy':<24}{'ROUGE-L':>9}{'margin':>10}{'clinF1':>9}{'firsts':>9}"
      f"{'uniqRep':>9}{'vocab':>8}{'words':>8}{'prior':>8}{'min':>7}")
print("  " + "-" * 101)
RES, OUTPUTS = {}, {}
for name, kw in STRATEGIES:
    t = time.time()
    try:
        P, R = run(SUB_DS, kw, name)
    except Exception as e:
        print(f"  {name:<24} FAILED: {type(e).__name__}: {str(e)[:60]}"); continue
    m = score(P, R); m["minutes"] = (time.time()-t)/60
    RES[name] = m; OUTPUTS[name] = (P, R)
    print(f"  {name:<24}{m['rougeL']:>9.4f}{m['margin']:>+10.4f}{m['clinical_f1']:>9.4f}"
          f"{m['uniq_firsts']:>9.4f}{m['uniq_reports']:>9.4f}{m['vocab_ratio']:>8.3f}"
          f"{m['gen_words']:>8.1f}{m['prior_rate']:>8.4f}{m['minutes']:>7.1f}")

_, R0 = OUTPUTS[list(OUTPUTS)[0]]
dr = diversity(R0)
print("  " + "-" * 101)
print(f"  {'REFERENCE (ceiling)':<24}{'—':>9}{'—':>10}{'—':>9}"
      f"{dr['uniq_firsts']:>9.4f}{dr['uniq_reports']:>9.4f}{1.0:>8.3f}{dr['words']:>8.1f}")
print("\n  The reference row is what REAL radiologists score on the same metric at the")
print("  same n. That — not 1.0 — is the ceiling any model can aim for.")

BEST_NAME = max(RES, key=lambda k: RES[k]["margin"])
print(f"\n  BEST BY MARGIN: {BEST_NAME}  ({RES[BEST_NAME]['margin']:+.4f})")

---
# 7 · The measurement-error fix — diversity at matched n

Stage 4 compared the old model at **n=100** against the new at **n=4,722**. That is
invalid: the ratio's denominator grew 47×. Here is the same model measured at
several n, with the reference alongside.

In [ ]:
P, R = OUTPUTS[BEST_NAME]
print("=" * 92); print(f"  DIVERSITY vs SAMPLE SIZE — {BEST_NAME}"); print("=" * 92)
print(f"  {'n':>7}{'GEN firsts':>13}{'REF firsts':>13}{'GEN uniqRep':>14}{'REF uniqRep':>14}{'vocab ratio':>13}")
print("  " + "-" * 74)
for n in (100, 250, 500, 1000):
    if n > len(P): break
    dg, drf = diversity(P[:n]), diversity(R[:n])
    print(f"  {n:>7}{dg['uniq_firsts']:>13.4f}{drf['uniq_firsts']:>13.4f}"
          f"{dg['uniq_reports']:>14.4f}{drf['uniq_reports']:>14.4f}"
          f"{dg['vocab']/max(drf['vocab'],1):>13.3f}")
d100 = diversity(P[:100])
print("\n  ⚠️  Diversity ratios FALL as n grows — for the model AND the references.")
print("     Comparing across different n is meaningless. This is the fair comparison:")
print(f"\n     OLD model  @ n=100 : uniq_firsts 0.1400   (from sample_reports_100.txt)")
print(f"     NEW model  @ n=100 : uniq_firsts {d100['uniq_firsts']:.4f}   ({BEST_NAME})")
print(f"     REFERENCES @ n=100 : uniq_firsts {diversity(R[:100])['uniq_firsts']:.4f}   (the ceiling)")

---
# 8 · Winner on the FULL test set

In [ ]:
kw = dict(STRATEGIES)[BEST_NAME]
print(f"  running {BEST_NAME} on all {len(FULL_DS):,} test samples ...")
t = time.time(); Pf, Rf = run(FULL_DS, kw, BEST_NAME); full = score(Pf, Rf)
print(f"  done in {(time.time()-t)/60:.1f} min\n")

print("=" * 92); print("  FINAL — FULL TEST SET"); print("=" * 92)
print(f"  {'metric':<32}{'Stage 4 (beam4)':>17}{'Stage 4B (' + BEST_NAME + ')':>26}{'Δ':>10}")
print("  " + "-" * 85)
for k, lab in [("rougeL", "ROUGE-L"), ("margin", "margin over baseline"),
               ("clinical_f1", "clinical-efficacy F1"), ("uniq_firsts", "unique firsts (n=4722)"),
               ("uniq_reports", "unique reports"), ("vocab_ratio", "vocabulary ratio"),
               ("prior_rate", "prior hallucination"), ("gen_words", "mean words")]:
    o = STAGE4_TEST.get(k, float("nan")); n_ = full[k]
    print(f"  {lab:<32}{o:>17.4f}{n_:>26.4f}{n_-o:>+10.4f}")
print("  " + "-" * 85)
print(f"\n  CONSTANT BASELINE = {CONST_BASELINE_ROUGEL:.4f}   MODEL = {full['rougeL']:.4f}"
      f"   MARGIN = {full['margin']:+.4f}")
print(f"  {'✅ clearly above baseline' if full['margin'] > 0.02 else ('✅ above baseline' if full['margin'] > 0.005 else '⚠️ marginal')}")

---
# 9 · Save

In [ ]:
out = {"stage": "4b", "timestamp": datetime.now().isoformat(),
       "checkpoint_epoch": int(ck["epoch"]), "best_strategy": BEST_NAME,
       "strategy_kwargs": {k: str(v) for k, v in kw.items()},
       "ablation_n": len(SUB_DS),
       "ablation": {k: {kk: float(vv) for kk, vv in v.items()} for k, v in RES.items()},
       "full_test": {k: float(v) for k, v in full.items()},
       "stage4_beam4_reference": STAGE4_TEST,
       "matched_n100": {"new_model": float(d100["uniq_firsts"]),
                        "old_model": 0.14,
                        "references": float(diversity(R[:100])["uniq_firsts"])},
       "constant_baseline_rougeL": CONST_BASELINE_ROUGEL,
       "hours": round((time.time()-T0)/3600, 2),
       "compute_units_est": round((time.time()-T0)/3600*RATE, 2)}
(REPORTS / "stage4b_ablation.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
with open(REPORTS / "stage4b_samples.txt", "w", encoding="utf-8") as f:
    for i in range(min(100, len(Pf))):
        f.write(f"--- [{i+1}]\n  REF: {' '.join(Rf[i].split())}\n  GEN: {' '.join(Pf[i].split())}\n\n")
print(f"  ✅ {REPORTS/'stage4b_ablation.json'}")
print(f"  ✅ {REPORTS/'stage4b_samples.txt'}")
print(f"\n  {(time.time()-T0)/3600:.2f} h ≈ {(time.time()-T0)/3600*RATE:.1f} CU")
print("\n  ⚠️  Runtime → Manage sessions → terminate.")